In [ ]:
import pandas as pd
from mplsoccer import Pitch ,VerticalPitch
import json
import numpy as np
import warnings
warnings.filterwarnings("ignore")

results_df = pd.read_csv('../../Our Datasets/classifier_results_rf.csv')

results_df.drop(columns={'Unique ID', 'event_index', 'source_file', 'row_index'}, inplace=True)

top_5_highest = results_df.sort_values('predicted_probability', ascending=False).head(5)

top_5_lowest = results_df.sort_values('predicted_probability', ascending=True).head(5)

In [ ]:
tracking_data = pd.read_csv('../../Our Datasets/processed_tracking_data_start.csv')

tracking_data.drop(columns={'is_detected_ball'}, inplace=True)

In [ ]:
tracking_data_highest = pd.merge(top_5_highest, tracking_data, left_on=['match_id', 'frame_anchor'], 
                                 right_on=['match_id', 'frame'], how
                                 ='left')

tracking_data_lowest = pd.merge(top_5_lowest, tracking_data, left_on=['match_id', 'frame_anchor'], 
                                 right_on=['match_id', 'frame'], how
                                 ='left')

tracking_data_highest[['frame_anchor', 'team_out_of_possession_phase_type']].drop_duplicates()

Just starting out with one for now since that's the most simple

## Great example of high goal scoring probability

In [ ]:
tracking_data_highest = tracking_data_highest.loc[tracking_data_highest['frame_anchor'] == 26953]

tracking_data_highest.reset_index(drop=True, inplace=True)

tracking_data_highest['rec_player_id'] = tracking_data_highest['rec_player_id'].astype(int)
tracking_data_highest['match_id'] = tracking_data_highest['match_id'].astype(int)

match_id = tracking_data_highest.loc[0, 'match_id']

We are getting the meta data so we can distinguish between the team in possession and the opponent

In [ ]:
def time_to_seconds(time_str):
    if time_str is None:
        return 90 * 60  # 120 minutes = 7200 seconds
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

file_path = f"../../data/matches/{match_id}/{match_id}_match.json"

with open(file_path, "r") as f:
    raw_match_data = json.load(f)

# The output has nested json elements. We process them
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
raw_match_df["home_team_side"] = raw_match_df["home_team_side"].astype(str)

players_df = pd.json_normalize(
    raw_match_df.to_dict("records"),
    record_path="players",
    meta=[
        "home_team_score",
        "away_team_score",
        "date_time",
        "home_team_side",
        "home_team.name",
        "home_team.id",
        "away_team.name",
        "away_team.id",
    ],  # data we keep
)


# Take only players who played and create their total time
players_df = players_df[
    ~((players_df.start_time.isna()) & (players_df.end_time.isna()))
]

# Create a flag for GK
players_df["is_gk"] = players_df["player_role.acronym"] == "GK"

# Add a flag if the given player is home or away
players_df["match_name"] = (
    players_df["home_team.name"] + " vs " + players_df["away_team.name"]
)


# Add a flag if the given player is home or away
players_df["home_away_player"] = np.where(
    players_df.team_id == players_df["home_team.id"], "Home", "Away"
)

# Create flag from player
players_df["team_name"] = np.where(
    players_df.team_id == players_df["home_team.id"],
    players_df["home_team.name"],
    players_df["away_team.name"],
)

# Figure out sides
players_df[["home_team_side_1st_half", "home_team_side_2nd_half"]] = (
    players_df["home_team_side"]
    .astype(str)
    .str.strip("[]")
    .str.replace("'", "")
    .str.split(", ", expand=True)
)
# Clean up sides
players_df["direction_player_1st_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_1st_half,
    players_df.home_team_side_2nd_half,
)
players_df["direction_player_2nd_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_2nd_half,
    players_df.home_team_side_1st_half,
)


# Clean up and keep the columns that we want to keep about

columns_to_keep = [
    "match_name",
    "home_team.name",
    "away_team.name",
    "id",
    "short_name",
    "team_id",
    "team_name",
    "player_role.position_group",
    "player_role.name",
    "player_role.acronym",
    "is_gk",
    "direction_player_1st_half",
    "direction_player_2nd_half",
]
players_df = players_df[columns_to_keep]

In [ ]:
enriched_tracking_data = tracking_data_highest.merge(
    players_df, left_on=["player_id"], right_on=["id"]
)

#enriched_tracking_data['rec_team_short'] = enriched_tracking_data['rec_team_short'] + ' ' + 'Football Club'


enriched_tracking_data["ball_carrier"] = enriched_tracking_data.rec_player_id == enriched_tracking_data.player_id
enriched_tracking_data["tip"] = enriched_tracking_data.rec_team_short == enriched_tracking_data.team_name

enriched_tracking_data.head()

In [ ]:
#synced["tip"] = synced.team_id_event == synced.team_id_tracking

pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

size = 300
possession_team = enriched_tracking_data[enriched_tracking_data.tip == True]
ax.scatter(
    possession_team["x"],
    possession_team["y"],
    c="#084D42",
    alpha=0.95,
    s=size,
    edgecolors="white",
    linewidths=1.5,
    zorder=10,
    label="team",
)

out_of_possession_team = enriched_tracking_data[enriched_tracking_data.tip == False]
ax.scatter(
    out_of_possession_team["x"],
    out_of_possession_team["y"],
    c="#E51717",
    alpha=0.95,
    s=size,
    edgecolors="white",
    linewidths=1,
    zorder=10,
    label="team",
)


ball_carrier = enriched_tracking_data[enriched_tracking_data.ball_carrier == True]
ax.scatter(
    ball_carrier["x"],
    ball_carrier["y"],
    c="#32FE6B",
    alpha=0.95,
    s=size,
    edgecolors="#32FE6B",
    linewidths=2.5,
    zorder=10,
    label="team",
)

b_size=150
ax.scatter(
    ball_carrier["ball_x"],
    ball_carrier["ball_y"],
    c="gray",
    alpha=0.95,
    s=b_size,
    edgecolors="gray",
    linewidths=2.5,
    zorder=10,
    label="team",
)

## Great example of low goal scoring recovery

In [ ]:
tracking_data_lowest[['frame_anchor', 'team_out_of_possession_phase_type']].drop_duplicates()

In [ ]:
tracking_data_lowest = tracking_data_lowest.loc[tracking_data_lowest['frame_anchor'] == 16056]

tracking_data_lowest.reset_index(drop=True, inplace=True)

tracking_data_lowest['rec_player_id'] = tracking_data_lowest['rec_player_id'].astype(int)
tracking_data_lowest['match_id'] = tracking_data_lowest['match_id'].astype(int)

match_id = tracking_data_lowest.loc[0, 'match_id']

We are getting the meta data so we can distinguish between the team in possession and the opponent

In [ ]:
def time_to_seconds(time_str):
    if time_str is None:
        return 90 * 60  # 120 minutes = 7200 seconds
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

file_path = f"../../data/matches/{match_id}/{match_id}_match.json"

with open(file_path, "r") as f:
    raw_match_data = json.load(f)

# The output has nested json elements. We process them
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
raw_match_df["home_team_side"] = raw_match_df["home_team_side"].astype(str)

players_df = pd.json_normalize(
    raw_match_df.to_dict("records"),
    record_path="players",
    meta=[
        "home_team_score",
        "away_team_score",
        "date_time",
        "home_team_side",
        "home_team.name",
        "home_team.id",
        "away_team.name",
        "away_team.id",
    ],  # data we keep
)


# Take only players who played and create their total time
players_df = players_df[
    ~((players_df.start_time.isna()) & (players_df.end_time.isna()))
]

# Create a flag for GK
players_df["is_gk"] = players_df["player_role.acronym"] == "GK"

# Add a flag if the given player is home or away
players_df["match_name"] = (
    players_df["home_team.name"] + " vs " + players_df["away_team.name"]
)


# Add a flag if the given player is home or away
players_df["home_away_player"] = np.where(
    players_df.team_id == players_df["home_team.id"], "Home", "Away"
)

# Create flag from player
players_df["team_name"] = np.where(
    players_df.team_id == players_df["home_team.id"],
    players_df["home_team.name"],
    players_df["away_team.name"],
)

# Figure out sides
players_df[["home_team_side_1st_half", "home_team_side_2nd_half"]] = (
    players_df["home_team_side"]
    .astype(str)
    .str.strip("[]")
    .str.replace("'", "")
    .str.split(", ", expand=True)
)
# Clean up sides
players_df["direction_player_1st_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_1st_half,
    players_df.home_team_side_2nd_half,
)
players_df["direction_player_2nd_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_2nd_half,
    players_df.home_team_side_1st_half,
)


# Clean up and keep the columns that we want to keep about

columns_to_keep = [
    "match_name",
    "home_team.name",
    "away_team.name",
    "id",
    "short_name",
    "team_id",
    "team_name",
    "player_role.position_group",
    "player_role.name",
    "player_role.acronym",
    "is_gk",
    "direction_player_1st_half",
    "direction_player_2nd_half",
]
players_df = players_df[columns_to_keep]

In [ ]:
enriched_tracking_data = tracking_data_lowest.merge(
    players_df, left_on=["player_id"], right_on=["id"]
)

#enriched_tracking_data['rec_team_short'] = enriched_tracking_data['rec_team_short'] + ' ' + 'Football Club'
enriched_tracking_data['rec_team_short'] = enriched_tracking_data['rec_team_short'].str.replace('Wellington P FC', 'Wellington Phoenix FC')

enriched_tracking_data["ball_carrier"] = enriched_tracking_data.rec_player_id == enriched_tracking_data.player_id
enriched_tracking_data["tip"] = enriched_tracking_data.rec_team_short == enriched_tracking_data.team_name

enriched_tracking_data.head()

In [ ]:
#synced["tip"] = synced.team_id_event == synced.team_id_tracking

pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

size = 300
possession_team = enriched_tracking_data[enriched_tracking_data.tip == True]
ax.scatter(
    possession_team["x"],
    possession_team["y"],
    c="#084D42",
    alpha=0.95,
    s=size,
    edgecolors="white",
    linewidths=1.5,
    zorder=10,
    label="team",
)

out_of_possession_team = enriched_tracking_data[enriched_tracking_data.tip == False]
ax.scatter(
    out_of_possession_team["x"],
    out_of_possession_team["y"],
    c="#E51717",
    alpha=0.95,
    s=size,
    edgecolors="white",
    linewidths=1,
    zorder=10,
    label="team",
)


ball_carrier = enriched_tracking_data[enriched_tracking_data.ball_carrier == True]
ax.scatter(
    ball_carrier["x"],
    ball_carrier["y"],
    c="#32FE6B",
    alpha=0.95,
    s=size,
    edgecolors="#32FE6B",
    linewidths=2.5,
    zorder=10,
    label="team",
)

b_size=150
ax.scatter(
    ball_carrier["ball_x"],
    ball_carrier["ball_y"],
    c="gray",
    alpha=0.95,
    s=b_size,
    edgecolors="gray",
    linewidths=2.5,
    zorder=10,
    label="team",
)